# Performer Attention Head Replacement in TinyLlama 1.1B Model - Analysis

This notebook benchmarks FAVOR+ (Performer) attention vs standard softmax attention inside TinyLlama 1.1B.

- **Section A** - Per-token quality comparison (KL divergence, token match)
- **Section B** - Speed benchmarks: B1 prefill O(N^2) vs O(NM), B2 decode O(N) vs O(M)
- **Section C** - Mixed-head quality sweep (0 to 32 performer heads)
- **Section D** - Independent text generation side-by-side
- **Section E** - Complexity convergence: E1 small-N regime, E2 M approaching N

## Setup

In [ ]:
!git clone https://github.com/Antoinechss/Performer-attention-LLM.git
!pip install -q transformers accelerate sentencepiece protobuf

# IMPORTANT: Do NOT %cd into the repo — the local transformers/ folder
# would shadow the pip-installed HuggingFace transformers package.
REPO_DIR = "/content/Performer-attention-LLM"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Triton available: ", end="")
try:
    import triton; print("Yes")
except ImportError:
    print("No")

In [ ]:
import sys, os, time
import torch
import torch.nn.functional as F

# Only add performer/ to path — NOT the repo root
sys.path.insert(0, os.path.join(REPO_DIR, 'performer'))
from performer_attention import PerformerAttentionCore, _HAS_TRITON

# ── Config ──────────────────────────────────────────────────────
MODEL          = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
PROMPT         = "<|user|>\nHow do I get a good night's sleep?</s>\n<|assistant|>\n"
MAX_NEW_TOKENS = 30
DTYPE          = torch.float16
DEVICE         = "cuda"

print(f"Triton kernel loaded: {_HAS_TRITON}")
print(f"Device: {DEVICE}, dtype: {DTYPE}")

In [ ]:
# ── Mixed Performer Attention wrapper (monkey-patches HF LlamaAttention) ────

class MixedPerformerAttention(torch.nn.Module):
    """Wraps a HuggingFace LlamaAttention, routing some heads through FAVOR+."""

    def __init__(self, original_attn, num_performer_heads):
        super().__init__()
        self.original = original_attn
        self.head_dim = original_attn.head_dim
        self.num_heads = original_attn.config.num_attention_heads
        self.num_key_value_heads = original_attn.config.num_key_value_heads
        self.num_key_value_groups = self.num_heads // self.num_key_value_heads
        self.scaling = self.head_dim ** -0.5
        self.num_performer_heads = num_performer_heads
        self.num_standard_heads = self.num_heads - num_performer_heads

        self.q_proj = original_attn.q_proj
        self.k_proj = original_attn.k_proj
        self.v_proj = original_attn.v_proj
        self.o_proj = original_attn.o_proj

        self.performer_core = PerformerAttentionCore(
            head_dim=self.head_dim, num_features=256
        ).to(DEVICE)

        self.config = original_attn.config
        self.layer_idx = original_attn.layer_idx
        self.is_causal = True

    def _rotate_half(self, x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def _apply_rotary(self, q, k, cos, sin):
        cos = cos.unsqueeze(1)
        sin = sin.unsqueeze(1)
        return (q * cos) + (self._rotate_half(q) * sin), (k * cos) + (self._rotate_half(k) * sin)

    def forward(self, hidden_states, position_embeddings=None,
                attention_mask=None, past_key_values=None, **kwargs):
        B, N, _ = hidden_states.shape

        q = self.q_proj(hidden_states).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(B, N, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(B, N, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        cos, sin = position_embeddings
        q, k = self._apply_rotary(q, k, cos, sin)

        if past_key_values is not None:
            k, v = past_key_values.update(k, v, self.layer_idx)

        if self.num_key_value_groups > 1:
            k = k.repeat_interleave(self.num_key_value_groups, dim=1)
            v = v.repeat_interleave(self.num_key_value_groups, dim=1)

        if self.num_standard_heads == 0:
            attn_out = self.performer_core(q, k, v)
        elif self.num_performer_heads == 0:
            scores = torch.matmul(q, k.transpose(-2, -1)) * self.scaling
            if attention_mask is not None:
                scores = scores + attention_mask
            w = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q.dtype)
            attn_out = torch.matmul(w, v)
        else:
            Kp = self.num_performer_heads
            out_p = self.performer_core(q[:, :Kp], k[:, :Kp], v[:, :Kp])

            q_s, k_s, v_s = q[:, Kp:], k[:, Kp:], v[:, Kp:]
            scores = torch.matmul(q_s, k_s.transpose(-2, -1)) * self.scaling
            if attention_mask is not None:
                scores = scores + attention_mask
            w = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q_s.dtype)
            out_s = torch.matmul(w, v_s)
            attn_out = torch.cat([out_p, out_s], dim=1)

        attn_out = attn_out.transpose(1, 2).contiguous().reshape(B, N, -1)
        return self.o_proj(attn_out), None

In [ ]:
# ── Load models ─────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
import copy

print("Classic Model Load")
std_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=DTYPE, device_map=DEVICE)
std_model.eval()

num_heads = std_model.config.num_attention_heads
### Tune number of attention heads replaced: ###
SECTION_A_PERFORMER_HEADS = 4

print("Performer Model Load")
perf_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=DTYPE, device_map=DEVICE)
perf_model.eval()

# Monkey-patch attention layers
for layer in perf_model.model.layers:
    layer.self_attn = MixedPerformerAttention(
        layer.self_attn, num_performer_heads=SECTION_A_PERFORMER_HEADS
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL)
prompt_ids = tokenizer(PROMPT, return_tensors="pt")["input_ids"].to(DEVICE)

print(f"Models loaded. Heads replaced: {SECTION_A_PERFORMER_HEADS}/{num_heads} performer.")

## Section A - Per-token quality comparison

Generates tokens step-by-step, comparing output distributions between standard and performer (4/32 performer heads). Tracks KL divergence and token match rate.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION A - Per-token generation comparison
# ═══════════════════════════════════════════════════════════════════

for layer in perf_model.model.layers:
    layer.self_attn.num_performer_heads = SECTION_A_PERFORMER_HEADS
    layer.self_attn.num_standard_heads  = num_heads - SECTION_A_PERFORMER_HEADS

print(f"\n{'='*70}")
print(f"SECTION A -- Generation  [{SECTION_A_PERFORMER_HEADS}/{num_heads} performer heads]")
print(f"{'='*70}\n")

W = 14
hdr = f"{'Step':>4}  {'Classic':.<{W}}  {'Performer':.<{W}}  {'p(cls)':>7}  {'p_perf(cls)':>11}  {'KL':>6}"
print(hdr)
print("-" * len(hdr))

current_ids = prompt_ids.clone()
classic_tokens, perf_tokens = [], []
kl_per_step = []

with torch.no_grad():
    for step in range(1, MAX_NEW_TOKENS + 1):
        std_out  = std_model(input_ids=current_ids,  use_cache=False)
        perf_out = perf_model(input_ids=current_ids, use_cache=False)

        std_logits  = std_out.logits[0, -1].float()
        perf_logits = perf_out.logits[0, -1].float()
        std_probs  = F.softmax(std_logits,  dim=-1)
        perf_probs = F.softmax(perf_logits, dim=-1)

        classic_id = std_logits.argmax().item()
        perf_id    = perf_logits.argmax().item()
        classic_p  = std_probs[classic_id].item()
        perf_p_cls = perf_probs[classic_id].item()
        kl         = F.kl_div(perf_probs.log(), std_probs, reduction='sum').item()

        c_tok = repr(tokenizer.decode([classic_id]))[1:-1]
        p_tok = repr(tokenizer.decode([perf_id]))[1:-1]
        print(f"{step:>4}  {c_tok:<{W}}  {p_tok:<{W}}  {classic_p:>6.1%}  {perf_p_cls:>11.1%}  {kl:>6.2f}")

        classic_tokens.append(classic_id)
        perf_tokens.append(perf_id)
        kl_per_step.append(kl)

        current_ids = torch.cat([current_ids, torch.tensor([[classic_id]], device=DEVICE)], dim=-1)
        if classic_id == tokenizer.eos_token_id:
            break

n = len(classic_tokens)
print(f"\n  Classic:   {tokenizer.decode(classic_tokens, skip_special_tokens=True)}")
print(f"  Performer: {tokenizer.decode(perf_tokens, skip_special_tokens=True)}")
match = sum(c == p for c, p in zip(classic_tokens, perf_tokens))
print(f"  Match: {match}/{n}  |  Avg KL: {sum(kl_per_step)/n:.3f}")

## Section B - Speed benchmarks (GPU + Triton)

**B1 - Prefill**: O(N^2) standard softmax vs O(N*M) FAVOR+ causal scan (Triton kernel).
Tests N = 256..8192 with M in {128, 256}.

**B2 - Decode**: O(N) standard KV lookup vs O(M) performer state query.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION B - Speed benchmarks (kernel-level)
# ═══════════════════════════════════════════════════════════════════

try:
    from triton_scan import triton_scan_forward as _triton_scan_raw
    from triton_scan import triton_decode_forward as _triton_decode_raw
except ImportError:
    _triton_scan_raw = _triton_decode_raw = None

_CUDA = torch.cuda.is_available()
_dev  = torch.device("cuda" if _CUDA else "cpu")
_TRITON = _HAS_TRITON and _CUDA

H, D     = 32, 64
M_VALS   = [128, 256]
REPEATS  = 10
scale    = D ** -0.25

performer_cores = {m: PerformerAttentionCore(head_dim=D, num_features=m).to(_dev) for m in M_VALS}

def time_fn(fn, repeats=REPEATS):
    # Warmup
    fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeats):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeats * 1000

# ── B1: Prefill ─────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"B1 -- Prefill  O(N^2) std vs O(N*M) performer  |  H={H} D={D}  [GPU + Triton={_TRITON}]")
print(f"{'='*70}\n")

SEQ_LENS = [256, 512, 1024, 2048, 4096, 8192]
_CW = 12

hdr = f"{'N':>6}  {'Std(ms)':>8}" + "".join(f"  {'e2e M='+str(m):>{_CW}}" for m in M_VALS) + f"  {'speedup':>8}"
print(hdr)
print("-" * len(hdr))

b1_results = []
with torch.no_grad():
    for N in SEQ_LENS:
        q = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16)
        k = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16)
        v = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16)

        def std_attn():
            w = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * (D ** -0.5), dim=-1)
            return torch.matmul(w, v)

        std_ms = time_fn(std_attn)
        row = f"{N:>6}  {std_ms:>7.2f}"

        e2e_times = {}
        for m in M_VALS:
            core = performer_cores[m]
            fn = lambda c=core: c(q, k, v)
            e2e_times[m] = time_fn(fn)
            row += f"  {e2e_times[m]:>{_CW}.2f}"

        best = min(e2e_times.values())
        speedup = std_ms / best
        row += f"  {speedup:>7.2f}x"
        print(row)
        b1_results.append((N, std_ms, e2e_times, speedup))

In [ ]:
# ── B2: Decode ──────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"B2 -- Decode step  O(N) std vs O(M) performer state  [GPU + Triton={_TRITON}]")
print(f"{'='*70}\n")

CACHE_SIZES = [64, 256, 1024, 4096]
_CW = 12

hdr2 = f"{'Cache':>6}  {'Std(ms)':>8}" + "".join(f"  {'M='+str(m):>{_CW}}" for m in M_VALS) + f"  {'speedup':>8}"
print(hdr2)
print("-" * len(hdr2))

with torch.no_grad():
    for N in CACHE_SIZES:
        q_new = torch.randn(1, H, 1, D, device=_dev, dtype=torch.float32)
        k_all = torch.randn(1, H, N, D, device=_dev, dtype=torch.float32)
        v_all = torch.randn(1, H, N, D, device=_dev, dtype=torch.float32)

        def std_decode():
            w = torch.softmax(torch.matmul(q_new, k_all.transpose(-2, -1)) * (D ** -0.5), dim=-1)
            return torch.matmul(w, v_all)

        std_ms = time_fn(std_decode)
        row = f"{N:>6}  {std_ms:>7.3f}"

        best_perf = float('inf')
        for m in M_VALS:
            core      = performer_cores[m]
            phi_k_all = core.phi(k_all * scale, is_query=False)
            kv_state  = torch.einsum("bhnm,bhnd->bhmd", phi_k_all, v_all).float()
            k_state   = phi_k_all.sum(dim=2).float()
            omega_m   = core.omega.float()

            if _TRITON and _triton_decode_raw is not None:
                fn = lambda kv=kv_state, ks=k_state, om=omega_m: \
                    _triton_decode_raw((q_new * scale).float(), om, kv, ks)
            else:
                def fn(kv=kv_state, ks=k_state, c=core):
                    phi_q = c.phi(q_new * scale, is_query=True)
                    out   = torch.einsum("bhnm,bhmd->bhnd", phi_q, kv)
                    denom = torch.einsum("bhnm,bhm->bhn", phi_q, ks) + 1e-6
                    return out / denom.unsqueeze(-1)

            t = time_fn(fn)
            best_perf = min(best_perf, t)
            row += f"  {t:>{_CW}.3f}"

        row += f"  {std_ms/best_perf:>7.2f}x"
        print(row)

## Section B - Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot B1: Prefill scaling
ax = axes[0]
ns = [r[0] for r in b1_results]
std_times = [r[1] for r in b1_results]
ax.plot(ns, std_times, 'o-', label='Standard O(N^2)', linewidth=2, color='tab:red')
for m in M_VALS:
    perf_times = [r[2][m] for r in b1_results]
    ax.plot(ns, perf_times, 's--', label=f'Performer M={m}', linewidth=2)
ax.set_xlabel('Sequence length N')
ax.set_ylabel('Time (ms)')
ax.set_title('B1: Prefill latency')
ax.legend()
ax.set_yscale('log')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

# Plot speedup
ax2 = axes[1]
speedups = [r[3] for r in b1_results]
ax2.bar([str(n) for n in ns], speedups, color=['tab:green' if s > 1 else 'tab:orange' for s in speedups])
ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, label='break-even')
ax2.set_xlabel('Sequence length N')
ax2.set_ylabel('Speedup (std / best performer)')
ax2.set_title('B1: Performer speedup factor')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Section C - Mixed-head quality sweep

Sweeps the number of performer heads from 0 (all standard) to 32 (all performer) and measures output divergence from the standard model.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION C - Mixed-head quality sweep
# ═══════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print("SECTION C -- Quality vs performer head count")
print(f"{'='*70}\n")

with torch.no_grad():
    std_ref = std_model(input_ids=prompt_ids, use_cache=False)
std_logits_ref = std_ref.logits[0, -1].float()
std_probs_ref  = F.softmax(std_logits_ref, dim=-1)
std_top5_ref   = set(std_logits_ref.topk(5).indices.tolist())

print(f"{'K':>4}  {'KL':>7}  {'Top5':>5}  {'p(top1)':>8}")
print("-" * 30)

c_results = []
for k in [0, 1, 2, 4, 8, 16, 32]:
    for layer in perf_model.model.layers:
        layer.self_attn.num_performer_heads = k
        layer.self_attn.num_standard_heads  = num_heads - k

    with torch.no_grad():
        out = perf_model(input_ids=prompt_ids, use_cache=False)

    logits  = out.logits[0, -1].float()
    probs   = F.softmax(logits, dim=-1)
    kl      = F.kl_div(probs.log(), std_probs_ref, reduction='sum').item()
    overlap = len(set(logits.topk(5).indices.tolist()) & std_top5_ref)
    p_top1  = probs[std_logits_ref.argmax().item()].item()

    print(f"{k:>4}  {kl:>7.3f}  {overlap:>4}/5  {p_top1:>8.1%}")
    c_results.append((k, kl, overlap, p_top1))

In [ ]:
# Plot Section C
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ks = [r[0] for r in c_results]
kls = [r[1] for r in c_results]
p_top1s = [r[3] for r in c_results]

ax = axes[0]
ax.plot(ks, kls, 'o-', linewidth=2, color='tab:red')
ax.set_xlabel('Number of performer heads (K)')
ax.set_ylabel('KL divergence from standard')
ax.set_title('C: Quality degradation vs performer heads')
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(ks, [p * 100 for p in p_top1s], 's-', linewidth=2, color='tab:blue')
ax2.set_xlabel('Number of performer heads (K)')
ax2.set_ylabel('p(standard top-1 token) %')
ax2.set_title('C: Probability assigned to standard model\'s top token')
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section D - Independent text generation

Each model generates freely from the same prompt. This shows actual generation quality side-by-side.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION D - Independent text generation
# ═══════════════════════════════════════════════════════════════════

GEN_TOKENS = 100

# Set performer to all-performer for maximum contrast
for layer in perf_model.model.layers:
    layer.self_attn.num_performer_heads = num_heads
    layer.self_attn.num_standard_heads  = 0

print(f"{'='*70}")
print(f"SECTION D -- Independent generation ({GEN_TOKENS} tokens, {num_heads}/{num_heads} performer heads)")
print(f"{'='*70}\n")
print(f"Prompt: {PROMPT.strip()}\n")

# Standard model generation
with torch.no_grad():
    std_ids = std_model.generate(
        prompt_ids, max_new_tokens=GEN_TOKENS, do_sample=False
    )
std_text = tokenizer.decode(std_ids[0], skip_special_tokens=True)

# Performer model generation (greedy, token-by-token since generate() may not work with patched attn)
with torch.no_grad():
    cur = prompt_ids.clone()
    for _ in range(GEN_TOKENS):
        out = perf_model(input_ids=cur, use_cache=False)
        next_id = out.logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
        cur = torch.cat([cur, next_id], dim=-1)
        if next_id.item() == tokenizer.eos_token_id:
            break
perf_text = tokenizer.decode(cur[0], skip_special_tokens=True)

print("--- Standard (softmax) ---")
print(std_text)
print(f"\n--- Performer (all {num_heads} heads FAVOR+) ---")
print(perf_text)

# Also show with partial performer heads
for k_test in [4, 8, 16]:
    for layer in perf_model.model.layers:
        layer.self_attn.num_performer_heads = k_test
        layer.self_attn.num_standard_heads  = num_heads - k_test
    with torch.no_grad():
        cur = prompt_ids.clone()
        for _ in range(GEN_TOKENS):
            out = perf_model(input_ids=cur, use_cache=False)
            next_id = out.logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
            cur = torch.cat([cur, next_id], dim=-1)
            if next_id.item() == tokenizer.eos_token_id:
                break
    text = tokenizer.decode(cur[0], skip_special_tokens=True)
    print(f"\n--- Performer ({k_test}/{num_heads} heads FAVOR+) ---")
    print(text)

## Section E — Complexity convergence analysis

Two experiments that confirm the theoretical O(N*M) vs O(N^2) scaling:

**E1 — Fix M, vary N (small N regime):** When N is small, the quadratic cost N^2 is cheap, so performer (which has overhead from phi() projection + scan) offers no speedup. The crossover point depends on M.

**E2 — Fix N, vary M toward N:** As M approaches N, the performer cost O(N*M) approaches O(N^2) — the same as standard attention. The speedup should converge to ~1x (or below, due to overhead).

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E1 — Fix M, sweep N from small to large
#       Shows: performer has no advantage at small N, crossover emerges
# ═══════════════════════════════════════════════════════════════════

E1_N_VALS = [32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]
E1_M_VALS = [64, 128, 256]

print(f"\n{'='*70}")
print(f"E1 -- Fix M, vary N  |  H={H} D={D}")
print(f"{'='*70}\n")

hdr = f"{'N':>6}  {'Std(ms)':>8}" + "".join(f"  {'M='+str(m):>10}" for m in E1_M_VALS)
print(hdr)
print("-" * len(hdr))

e1_results = []  # list of (N, std_ms, {m: perf_ms})

with torch.no_grad():
    for N in E1_N_VALS:
        q = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16)
        k = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16)
        v = torch.randn(1, H, N, D, device=_dev, dtype=torch.float16)

        def std_attn():
            w = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * (D ** -0.5), dim=-1)
            return torch.matmul(w, v)

        std_ms = time_fn(std_attn)
        row = f"{N:>6}  {std_ms:>7.2f}"

        perf_times = {}
        for m in E1_M_VALS:
            if m not in performer_cores:
                performer_cores[m] = PerformerAttentionCore(head_dim=D, num_features=m).to(_dev)
            core = performer_cores[m]
            fn = lambda c=core: c(q, k, v)
            perf_times[m] = time_fn(fn)
            row += f"  {perf_times[m]:>9.2f}"

        print(row)
        e1_results.append((N, std_ms, perf_times))

print("\nExpected: at small N, performer is slower (overhead > N^2 savings)."
      "\nAt large N, performer wins because N*M << N^2.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E2 — Fix N, sweep M from small toward N
#       Shows: as M -> N, performer cost -> standard cost (speedup -> 1x)
# ═══════════════════════════════════════════════════════════════════

E2_N = 2048
E2_M_VALS = [32, 64, 128, 256, 512, 1024, 2048]

print(f"\n{'='*70}")
print(f"E2 -- Fix N={E2_N}, vary M toward N  |  H={H} D={D}")
print(f"{'='*70}\n")

# Standard attention baseline for this N
with torch.no_grad():
    q = torch.randn(1, H, E2_N, D, device=_dev, dtype=torch.float16)
    k = torch.randn(1, H, E2_N, D, device=_dev, dtype=torch.float16)
    v = torch.randn(1, H, E2_N, D, device=_dev, dtype=torch.float16)

    def std_attn():
        w = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * (D ** -0.5), dim=-1)
        return torch.matmul(w, v)

    e2_std_ms = time_fn(std_attn)

print(f"Standard attention at N={E2_N}: {e2_std_ms:.2f} ms\n")
print(f"{'M':>6}  {'M/N':>6}  {'Perf(ms)':>9}  {'Speedup':>8}  {'Ratio':>8}")
print("-" * 45)

e2_results = []  # list of (M, perf_ms, speedup)

with torch.no_grad():
    for m in E2_M_VALS:
        if m not in performer_cores:
            performer_cores[m] = PerformerAttentionCore(head_dim=D, num_features=m).to(_dev)
        core = performer_cores[m]
        fn = lambda c=core: c(q, k, v)
        perf_ms = time_fn(fn)
        speedup = e2_std_ms / perf_ms
        ratio = m / E2_N

        print(f"{m:>6}  {ratio:>5.2f}  {perf_ms:>8.2f}  {speedup:>7.2f}x  {'<-- M=N' if m == E2_N else ''}")
        e2_results.append((m, perf_ms, speedup))

print(f"\nExpected: speedup decreases as M approaches N."
      f"\nAt M=N={E2_N}, performer does O(N*N)=O(N^2) — same as standard, plus phi() overhead.")

## Section E — Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── E1 Plot 1: Latency curves (log-log) ────────────────────────
ax = axes[0, 0]
e1_ns = [r[0] for r in e1_results]
e1_std = [r[1] for r in e1_results]
ax.plot(e1_ns, e1_std, 'o-', label='Standard O(N^2)', linewidth=2, color='tab:red')
colors = ['tab:blue', 'tab:green', 'tab:purple']
for i, m in enumerate(E1_M_VALS):
    perf = [r[2][m] for r in e1_results]
    ax.plot(e1_ns, perf, 's--', label=f'Performer M={m}', linewidth=2, color=colors[i])
ax.set_xlabel('Sequence length N')
ax.set_ylabel('Time (ms)')
ax.set_title('E1: Latency vs N (log-log) — small N = no advantage')
ax.legend(fontsize=9)
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# ── E1 Plot 2: Speedup vs N (shows crossover point) ────────────
ax = axes[0, 1]
for i, m in enumerate(E1_M_VALS):
    speedups = [r[1] / r[2][m] for r in e1_results]
    ax.plot(e1_ns, speedups, 'o-', label=f'M={m}', linewidth=2, color=colors[i])
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, linewidth=1, label='break-even')
ax.set_xlabel('Sequence length N')
ax.set_ylabel('Speedup (std / performer)')
ax.set_title('E1: Speedup vs N — crossover from <1x to >1x')
ax.legend(fontsize=9)
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

# ── E2 Plot 1: Latency vs M (approaching N) ────────────────────
ax = axes[1, 0]
e2_ms = [r[0] for r in e2_results]
e2_perf = [r[1] for r in e2_results]
ax.plot(e2_ms, e2_perf, 's-', label='Performer', linewidth=2, color='tab:blue')
ax.axhline(y=e2_std_ms, color='tab:red', linewidth=2, label=f'Standard (N={E2_N})')
ax.set_xlabel(f'Number of features M (N={E2_N})')
ax.set_ylabel('Time (ms)')
ax.set_title('E2: Performer latency converges to standard as M -> N')
ax.legend(fontsize=9)
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)

# ── E2 Plot 2: Speedup vs M/N ratio ────────────────────────────
ax = axes[1, 1]
e2_ratios = [m / E2_N for m in e2_ms]
e2_speedups = [r[2] for r in e2_results]
ax.plot(e2_ratios, e2_speedups, 'o-', linewidth=2, color='tab:green', markersize=8)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, linewidth=1, label='break-even')
ax.set_xlabel('M / N ratio')
ax.set_ylabel('Speedup (std / performer)')
ax.set_title('E2: Speedup converges to ~1x (or below) as M/N -> 1')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)
# Mark key points
for ratio, sp in zip(e2_ratios, e2_speedups):
    if ratio in [0.0625, 0.5, 1.0]:
        ax.annotate(f'{sp:.1f}x', (ratio, sp), textcoords="offset points",
                    xytext=(10, 5), fontsize=9)

plt.suptitle('Section E: Complexity convergence — O(N*M) vs O(N^2)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()